# EDA NLP Text

Exploratory analysis for NLP text datasets (fake news).

Steps:
- Verify data presence and schema.
- Inspect label balance and missing values.
- Summarize text length and word counts.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

nlp_root = REPO_ROOT / 'data' / 'raw' / 'nlp' / 'fakenews'
dataset_path = nlp_root / 'fake_news_labeled.csv'

summary = {
    'dataset': {},
    'text_stats': {},
}

print('NLP root:', nlp_root)
if not nlp_root.exists():
    print('Missing:', nlp_root)
else:
    for child in sorted(nlp_root.iterdir()):
        print(' -', child.name)

if dataset_path.exists():
    summary['dataset']['path'] = str(dataset_path)
    summary['dataset']['size_mb'] = round(dataset_path.stat().st_size / 1024**2, 2)
    print('Dataset size (MB):', summary['dataset']['size_mb'])
else:
    print('Missing dataset:', dataset_path)


In [ ]:
# Sample the dataset to inspect schema and labels.
if dataset_path.exists():
    df = pd.read_csv(dataset_path, nrows=50000)
    summary['dataset']['rows_sampled'] = int(df.shape[0])
    summary['dataset']['columns'] = list(df.columns)
    print('Sample shape:', df.shape)
    print('Columns:', list(df.columns))

    missing = df.isna().sum().sort_values(ascending=False).head(10).to_dict()
    summary['dataset']['missing_top'] = missing
    print('Missing values (top 10):', missing)

    label_candidates = ['label', 'class', 'target', 'is_fake', 'fake', 'is_fake_news']
    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col:
        counts = df[label_col].value_counts(dropna=False).to_dict()
        summary['dataset']['label_col'] = label_col
        summary['dataset']['label_counts'] = counts
        print('Label distribution:', counts)
    else:
        print('No label column found in sample')

    text_candidates = [col for col in df.columns if df[col].dtype == 'object']
    preferred = [col for col in text_candidates if any(key in col.lower() for key in ['text', 'content', 'title'])]
    text_col = preferred[0] if preferred else (text_candidates[0] if text_candidates else None)
    if text_col:
        text_series = df[text_col].fillna('').astype(str)
        lengths = text_series.str.len()
        word_counts = text_series.str.split().map(len)
        summary['text_stats']['text_col'] = text_col
        summary['text_stats']['length_desc'] = lengths.describe().to_dict()
        summary['text_stats']['word_desc'] = word_counts.describe().to_dict()
        print('Text column:', text_col)
        print('Length stats:', lengths.describe())
        print('Word count stats:', word_counts.describe())
    else:
        print('No text column found in sample')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_nlp_text_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize nlp-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'nlp' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No nlp entries found in TRAINING_DATA.json')
    else:
        print('nlp datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
